# GO-7, one click: description rate and reset work are different resources

**Claim** ([`Paper V`](../paper/consumer-relative-landauer.pdf), Theorem 1; ledger row **GO-7**, class `[replicated]`):
the *same* stored code index needs ~R bits/symbol to **describe** to a consumer, but its
**reset residual** — what a mechanism holding correlated side information must still
irreversibly clear — is decodable from the side information at a bin rate near the
*conditional* content I(X;X_hat|S), a fraction of R. Below that content, binning fails;
without the side information, it fails at any rate.

This notebook re-runs the binary instance (sealed preregistrations
[GO-P-2026-043](../prereg/GO-P-2026-043-landauer-operational-separation.md),
[045](../prereg/GO-P-2026-045-landauer-multicodebook.md)) at reduced blocklengths —
CPU-only, numpy-only, ~2 minutes. The full governed runs and their committed JSONs are in
[`results/`](../results/).

**Replication call** (PROTOCOL §10): run it, change `SEED`, change the target channel —
if you can make the separation disappear while the gates' preconditions hold, that is a
`[refuted]`-grade finding; please open an issue.


In [ ]:
# GO-7 reduced re-run: numpy only, ~2 min on CPU
import numpy as np
import matplotlib.pyplot as plt

SEED = 20260802               # change me for a fresh replication
DA, DB = 0.08, 0.32           # target channel BSC(DA) x BSC(DB); S = A
h2 = lambda t: 0.0 if t <= 0 or t >= 1 else -t*np.log2(t)-(1-t)*np.log2(1-t)
R_AN = 2 - h2(DA) - h2(DB); L_AN = 1 - h2(DB)
WA, WB = np.log((1-DA)/DA), np.log((1-DB)/DB)
LUT = np.array([bin(i).count("1") for i in range(65536)], dtype=np.uint16)
pop = lambda a: LUT[np.ascontiguousarray(a).view(np.uint16).reshape(-1,4)].sum(1, dtype=np.int64)

rng = np.random.default_rng(SEED)
rbs = [0.03, 0.08, 0.13, 0.19, 0.26, 0.35, 0.50]
print(f"analytic R={R_AN:.3f}  L=I(X;Xh|S)={L_AN:.3f}  (bits/symbol)")
curves = {}
for n, T in [(12, 150), (16, 150), (20, 150)]:
    nb = int(np.ceil(n*(R_AN+0.03))); Ncw = 1 << nb
    cwA = rng.integers(0, 1<<n, Ncw, dtype=np.uint64)
    cwB = rng.integers(0, 1<<n, Ncw, dtype=np.uint64)
    xA = rng.integers(0, 1<<n, T, dtype=np.uint64)
    xB = rng.integers(0, 1<<n, T, dtype=np.uint64)
    M = np.array([int(np.argmin(WA*pop(cwA^xA[t]) + WB*pop(cwB^xB[t]))) for t in range(T)])
    errs, errs_no_si = [], []
    for rb in rbs:
        nbins = 1 << max(1, int(np.ceil(n*rb))); ok = ok0 = 0
        for t in range(T):
            mem = np.arange(int(M[t]) % nbins, Ncw, nbins, dtype=np.int64)
            sc = pop(cwA[mem] ^ xA[t])          # ML from side info S^n = A^n
            ok += int(mem[np.argmin(sc)]) == M[t]
            ok0 += int(mem[rng.integers(0, mem.size)]) == M[t]   # no-SI control
        errs.append(1-ok/T); errs_no_si.append(1-ok0/T)
    curves[n] = (errs, errs_no_si)
    print(f"n={n:2d}: err(SI) by bin rate {np.round(errs,2)}   no-SI {np.round(errs_no_si,2)}")

plt.figure(figsize=(7,4.2))
for n,(e,_) in curves.items(): plt.plot(rbs, e, "o-", label=f"with $S^n$, n={n}")
plt.plot(rbs, curves[20][1], "x--", color="gray", label="no side info (n=20)")
plt.axvline(L_AN, ls=":", color="k"); plt.text(L_AN+.005, .8, r"$I(X;\hat X|S)$")
plt.axvline(R_AN, ls=":", color="r"); plt.text(R_AN-.11, .8, r"$I(X;\hat X)$", color="r")
plt.xlabel("bin rate $r_b$ (bits/symbol)"); plt.ylabel("recovery error")
plt.title("Same stored index: describable at R, resettable at ~L"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
# gates (reduced-n analogues of the sealed bars)
e20, e20_ctrl = curves[20]
assert e20[rbs.index(0.26)] <= 0.15, "separation should decode at rb=0.26"
assert e20[rbs.index(0.03)] >= 0.30, "below-content binning should fail"
assert min(e20_ctrl[:5]) >= 0.90,   "no-SI control should fail"
print("GO-7 reduced-n gates: PASS -- the separation is yours to try to break.")
